In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, make_column_selector, make_column_transformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.impute import SimpleImputer
from lightgbm import LGBMRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    StackingRegressor,
    GradientBoostingRegressor,
    VotingRegressor
)
from xgboost import XGBRegressor
import os

In [10]:
train = pd.read_csv("D:/Machine_Learning/AV_Big_Mart/train_v9rqX0R.csv")
test = pd.read_csv("D:/Machine_Learning/AV_Big_Mart/test_AbJTz2l.csv")
submission = pd.read_csv("D:/Machine_Learning/AV_Big_Mart/sample_submission_8RXa3c6.csv")

train.head()

,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Outlet_Type,Item_Outlet_Sales
0,FDA15,9.30,Low Fat,0.016047,Dairy,249.8092,OUT049,1999,Medium,Tier 1,Supermarket Type1,3735.1380
1,DRC01,5.92,Regular,0.019278,Soft Drinks,48.2692,OUT018,2009,Medium,Tier 3,Supermarket Type2,443.4228
2,FDN15,17.50,Low Fat,0.016760,Meat,141.6180,OUT049,1999,Medium,Tier 1,Supermarket Type1,2097.2700
3,FDX07,19.20,Regular,0.000000,Fruits and Vegetables,182.0950,OUT010,1998,NaN,Tier 3,Grocery Store,732.3800
4,NCD19,8.93,Low Fat,0.000000,Household,53.8614,OUT013,1987,High,Tier 3,Supermarket Type1,994.7052


In [11]:
train.columns

Index(['Item_Identifier', 'Item_Weight', 'Item_Fat_Content', 'Item_Visibility',
       'Item_Type', 'Item_MRP', 'Outlet_Identifier',
       'Outlet_Establishment_Year', 'Outlet_Size', 'Outlet_Location_Type',
       'Outlet_Type', 'Item_Outlet_Sales'],
      dtype='object')

In [12]:
X_train = train.drop(['Item_Identifier', 'Outlet_Identifier', 'Item_Outlet_Sales'], axis=1)  # removed columns bcz after ohe it makes 1200+ columns
X_test = test.drop(['Item_Identifier', 'Outlet_Identifier'], axis=1)

In [13]:
X_train.isnull().sum()

Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
dtype: int64

In [6]:
# imp_mode = SimpleImputer(strategy='most_frequent')
# imp_med = SimpleImputer(strategy='median')

ohc = OneHotEncoder(sparse_output=False)
# ct_imp = make_column_transformer((imp_mode, 
#                                   make_column_selector(dtype_include=object)),
#                                 (imp_med, 
#                                  make_column_selector(dtype_include=['int64', 'float64'])),
#                                 verbose_feature_names_out=False).set_output(transform="pandas")

ct_enc = make_column_transformer((ohc,
                                 make_column_selector(dtype_include=object)),
                                ("passthrough", 
                                make_column_selector(dtype_include=['int64', 'float64'])),
                                verbose_feature_names_out=False).set_output(transform="pandas") 

# pipe = Pipeline([('IMP', ct_imp), ('OHE', ct_enc)])

# X_trn_prc = pipe.fit_transform(X_train)

In [7]:
X_trn_prc.shape

NameError: name 'X_trn_prc' is not defined

### Using XGBoost

In [ ]:
xgb = XGBRegressor(random_state=26, verbosity=0, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=26)

pipe_xgb = Pipeline([('IMP', ct_imp), ('OHE', ct_enc), ('XGB', xgb)])

params_xgb = {
    'XGB__n_estimators'  : [50, 100, 200],
    'XGB__learning_rate' : [0.05, 0.1, 0.2],
    'XGB__max_depth'     : [3, 4, 6]
}

gcv_xgb = GridSearchCV(
    pipe_xgb, param_grid=params_xgb,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_xgb.fit(X_train,test)

print(f"\nBest CV R² (XGBoost) : {gcv_xgb.best_score_:.4f}")
print(f"Best Params          : {gcv_xgb.best_params_}")

In [ ]:
# --- BOOSTING Submission ---
y_pred_boost = gcv_xgb.best_estimator_.predict(test)
y_pred_boost = np.clip(y_pred_boost, 0, None)

sub_boost = submission.copy()
sub_boost['Item_Outlet_Sales'] = y_pred_boost
sub_boost.to_csv('"D:/Machine_Learning/AV_Big_Mart/submission_boosting.csv', index=False)
print("submission_boosting.csv saved ✓")

### Using LGBMRegressor

In [ ]:
# --- 7a. Gradient Boosting ---
gbm = GradientBoostingRegressor(random_state=26)
pipe_gbm = Pipeline([('IMP', ct_imp), ('OHE', ct_enc), ('GBM', gbm)])

params_gbm = {
    'GBM__n_estimators'  : [50, 100, 200],
    'GBM__learning_rate' : [0.05, 0.1, 0.2],
    'GBM__max_depth'     : [3, 4, 5,6]
}

gcv_gbm = GridSearchCV(
    pipe_gbm, param_grid=params_gbm,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_gbm.fit(X, y)

print(f"\nBest CV R² (GBM)  : {gcv_gbm.best_score_:.4f}")
print(f"Best Params       : {gcv_gbm.best_params_}")

In [ ]:
# --- BOOSTING Submission ---
y_pred_boost = gcv_xgb.best_estimator_.predict(test)
y_pred_boost = np.clip(y_pred_boost, 0, None)

sub_boost = submission.copy()
sub_boost['Item_Outlet_Sales'] = y_pred_boost
sub_boost.to_csv("D:/Machine_Learning/AV_Big_Mart/submission_GBMboosting.csv", index=False)
print("submission_boosting.csv saved ✓")

### Apply EDA on item_weight column to fill na values

In [14]:
item_weight_map = train.groupby('Item_Identifier')['Item_Weight'].mean()
item_weight_map

Item_Identifier
DRA12    11.600
DRA24    19.350
DRA59     8.270
DRB01     7.390
DRB13     6.115
          ...  
NCZ30     6.590
NCZ41    19.850
NCZ42    10.500
NCZ53     9.600
NCZ54    14.650
Name: Item_Weight, Length: 1559, dtype: float64

In [15]:
# 1. Map weights based on Item_Identifier
item_weight_map = train.groupby('Item_Identifier')['Item_Weight'].mean()

# 2. Fill NaNs using the mapping
train['Item_Weight'] = train['Item_Weight'].fillna(train['Item_Identifier'].map(item_weight_map))
test['Item_Weight'] = test['Item_Weight'].fillna(test['Item_Identifier'].map(item_weight_map))

# 3. Handle residual NaNs (items that have NO weight data at all) using Item_Type mean
# Handle residual NaNs using Item_Type mean
train['Item_Weight'] = train['Item_Weight'].fillna(train.groupby('Item_Type')['Item_Weight'].transform('mean'))
test['Item_Weight'] = test['Item_Weight'].fillna(test.groupby('Item_Type')['Item_Weight'].transform('mean'))



In [16]:
train.isnull().sum()

Item_Identifier                 0
Item_Weight                     0
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Identifier               0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
Item_Outlet_Sales               0
dtype: int64

In [17]:
#  Sir's way


### change item_fat to same names

In [18]:
# Check unique values before cleaning
print("Before cleaning (Train):", train['Item_Fat_Content'].unique())

# Standardize the labels
fat_content_fix = {
    'LF': 'Low Fat',
    'low fat': 'Low Fat',
    'reg': 'Regular'
}

train['Item_Fat_Content'] = train['Item_Fat_Content'].replace(fat_content_fix)
test['Item_Fat_Content'] = test['Item_Fat_Content'].replace(fat_content_fix)

# Verify the changes
print("After cleaning (Train):", train['Item_Fat_Content'].unique())
print("After cleaning (Test):", test['Item_Fat_Content'].unique())

Before cleaning (Train): ['Low Fat' 'Regular' 'low fat' 'LF' 'reg']
After cleaning (Train): ['Low Fat' 'Regular']
After cleaning (Test): ['Low Fat' 'Regular']


## now fit model

In [ ]:
test

In [19]:
X_train.isnull().sum()

Item_Weight                  1463
Item_Fat_Content                0
Item_Visibility                 0
Item_Type                       0
Item_MRP                        0
Outlet_Establishment_Year       0
Outlet_Size                  2410
Outlet_Location_Type            0
Outlet_Type                     0
dtype: int64

In [20]:
X_train = train.drop(['Item_Identifier', 'Outlet_Identifier','Item_Outlet_Sales'], axis=1)  # removed columns bcz after ohe it makes 1200+ columns
X_test = test.drop(['Item_Identifier', 'Outlet_Identifier'], axis=1)
X_train.isnull().sum(), X_test.isnull().sum()

(Item_Weight                     0
 Item_Fat_Content                0
 Item_Visibility                 0
 Item_Type                       0
 Item_MRP                        0
 Outlet_Establishment_Year       0
 Outlet_Size                  2410
 Outlet_Location_Type            0
 Outlet_Type                     0
 dtype: int64,
 Item_Weight                     0
 Item_Fat_Content                0
 Item_Visibility                 0
 Item_Type                       0
 Item_MRP                        0
 Outlet_Establishment_Year       0
 Outlet_Size                  1606
 Outlet_Location_Type            0
 Outlet_Type                     0
 dtype: int64)

In [ ]:
X_test

In [21]:

pipe = Pipeline([('OHE', ct_enc)])

X_trn_prc = pipe.fit_transform(X_train)
X_trn_prc.shape

(8523, 33)

In [23]:
y = train['Item_Outlet_Sales']

In [24]:
xgb = XGBRegressor(random_state=26, verbosity=0, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=26)

pipe_xgb = Pipeline([('OHE', ct_enc), ('XGB', xgb)])

params_xgb = {
    'XGB__n_estimators'  : [50, 100, 200],
    'XGB__learning_rate' : [0.05, 0.1, 0.2],
    'XGB__max_depth'     : [3, 4, 6]
}

gcv_xgb = GridSearchCV(
    pipe_xgb, param_grid=params_xgb,
    cv=kfold, scoring='r2', n_jobs=-1, verbose=1
)
gcv_xgb.fit(X_train, y)

print(f"\nBest CV R² (XGBoost) : {gcv_xgb.best_score_:.4f}")
print(f"Best Params          : {gcv_xgb.best_params_}")

Fitting 5 folds for each of 27 candidates, totalling 135 fits

Best CV R² (XGBoost) : 0.5999
Best Params          : {'XGB__learning_rate': 0.05, 'XGB__max_depth': 3, 'XGB__n_estimators': 100}


In [25]:
# --- BOOSTING Submission ---
y_pred_boost = gcv_xgb.best_estimator_.predict(X_test)
y_pred_boost = np.clip(y_pred_boost, 0, None)

sub_boost = submission.copy()
sub_boost['Item_Outlet_Sales'] = y_pred_boost
sub_boost.to_csv('D:/Machine_Learning/AV_Big_Mart/submission_boosting.csv', index=False)
print("submission_boosting.csv saved ✓")

submission_boosting.csv saved ✓
